In [ ]:
from pathlib import Path

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "python").is_dir() and (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from the "
        "credit-risk-analytics project folder or one of its subfolders."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
LOCAL_DATA_DIR = PROJECT_ROOT / "local_data" / "processed"
LOCAL_OUTPUT_DIR = PROJECT_ROOT / "local_outputs"

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_DIR)
print("Local processed-data folder:", LOCAL_DATA_DIR)
print("Local output folder:", LOCAL_OUTPUT_DIR)


In [ ]:
# CreditWise: Credit Risk Analytics Project
# Notebook 02: Data Cleaning and Feature Engineering

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

train_path = DATA_DIR / "application_train.csv"
test_path = DATA_DIR / "application_test.csv"
columns_path = DATA_DIR / "HomeCredit_columns_description.csv"

missing = [p.name for p in (train_path, test_path, columns_path) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing dataset file(s) in {DATA_DIR}: {missing}. "
        "See the project README for data setup instructions."
    )

print("Training file:", train_path.name)
print("Test file:", test_path.name)
print("Column description file:", columns_path.name)


In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
columns_desc = pd.read_csv(columns_path, encoding="latin1")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)
print("Columns description shape:", columns_desc.shape)


In [ ]:
train_clean = train.copy()
test_clean = test.copy()

print("Training copy shape:", train_clean.shape)
print("Test copy shape:", test_clean.shape)

In [ ]:
## 1. Cleaning Strategy

print("""The purpose of this notebook is to prepare the Home Credit dataset for statistical modelling and machine learning. The cleaning process focuses on correcting known anomalies, creating interpretable features, handling rare categories, managing missing values, and preparing a final modelling dataset.

The cleaning decisions are based on findings from Notebook 01. In particular, the DAYS_EMPLOYED variable contains an unrealistic placeholder value of 365243, missing data is substantial, and several variables require transformation to become more interpretable for credit risk modelling.""")

In [ ]:
# Identify the anomalous placeholder value in DAYS_EMPLOYED
anomaly_value = 365243

for data in [train_clean, test_clean]:
    data["DAYS_EMPLOYED_ANOMALY"] = (data["DAYS_EMPLOYED"] == anomaly_value).astype(int)
    data["DAYS_EMPLOYED_CLEAN"] = data["DAYS_EMPLOYED"].replace(anomaly_value, np.nan)
    data["YEARS_EMPLOYED"] = (-data["DAYS_EMPLOYED_CLEAN"]) / 365.25

train_clean[
    ["DAYS_EMPLOYED", "DAYS_EMPLOYED_ANOMALY", "DAYS_EMPLOYED_CLEAN", "YEARS_EMPLOYED"]
].describe().T

In [ ]:
print("""The value 365243 in DAYS_EMPLOYED is treated as an anomalous placeholder rather than a valid employment duration. A binary anomaly indicator was created to preserve the information that the value was originally present. The placeholder value was then replaced with missing values, and employment duration was converted into years for easier interpretation.  """)

In [ ]:
for data in [train_clean, test_clean]:
    data["AGE_YEARS"] = (-data["DAYS_BIRTH"]) / 365.25
    
    data["CREDIT_TO_INCOME_RATIO"] = data["AMT_CREDIT"] / data["AMT_INCOME_TOTAL"]
    data["ANNUITY_TO_INCOME_RATIO"] = data["AMT_ANNUITY"] / data["AMT_INCOME_TOTAL"]
    data["GOODS_TO_CREDIT_RATIO"] = data["AMT_GOODS_PRICE"] / data["AMT_CREDIT"]

new_features = [
    "AGE_YEARS",
    "YEARS_EMPLOYED",
    "CREDIT_TO_INCOME_RATIO",
    "ANNUITY_TO_INCOME_RATIO",
    "GOODS_TO_CREDIT_RATIO",
    "DAYS_EMPLOYED_ANOMALY"
]

train_clean[new_features].describe().T

In [ ]:
# Replace rare or unknown categorical values with missing values

for data in [train_clean, test_clean]:
    data["CODE_GENDER"] = data["CODE_GENDER"].replace("XNA", np.nan)
    data["NAME_FAMILY_STATUS"] = data["NAME_FAMILY_STATUS"].replace("Unknown", np.nan)

print("CODE_GENDER values:")
print(train_clean["CODE_GENDER"].value_counts(dropna=False))

print("\nNAME_FAMILY_STATUS values:")
print(train_clean["NAME_FAMILY_STATUS"].value_counts(dropna=False))

In [ ]:
print("""## 2. Feature Selection for the First Model

For the first modelling dataset, a focused set of variables is selected based on credit risk relevance, interpretability and findings from the exploratory analysis. The aim is to build a clean baseline dataset before introducing more complex feature engineering. """)

In [ ]:
# Select features for the first modelling dataset

selected_features = [
    # Target and ID
    "SK_ID_CURR",
    "TARGET",
    
    # Applicant demographics
    "CODE_GENDER",
    "AGE_YEARS",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "NAME_FAMILY_STATUS",
    "NAME_EDUCATION_TYPE",
    
    # Income and employment
    "NAME_INCOME_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "YEARS_EMPLOYED",
    "DAYS_EMPLOYED_ANOMALY",
    
    # Loan and affordability
    "NAME_CONTRACT_TYPE",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CREDIT_TO_INCOME_RATIO",
    "ANNUITY_TO_INCOME_RATIO",
    "GOODS_TO_CREDIT_RATIO",
    
    # Asset ownership and housing
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "NAME_HOUSING_TYPE",
    
    # External credit scores
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

model_data = train_clean[selected_features].copy()

print("Model dataset shape:", model_data.shape)
model_data.head()

In [ ]:
# Missing values in selected modelling features

model_missing = pd.DataFrame({
    "Missing Count": model_data.isnull().sum(),
    "Missing Percentage": model_data.isnull().mean() * 100
}).sort_values(by="Missing Percentage", ascending=False)

model_missing

In [ ]:
# Identify numeric and categorical features in the modelling dataset

target_col = "TARGET"
id_col = "SK_ID_CURR"

numeric_features = model_data.drop(columns=[target_col, id_col]).select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = model_data.drop(columns=[target_col, id_col]).select_dtypes(include=["object"]).columns.tolist()

print("Number of numeric features:", len(numeric_features))
print(numeric_features)

print("\nNumber of categorical features:", len(categorical_features))
print(categorical_features)

In [ ]:
print("""## 3. Missing Value Treatment

Missing values are treated according to the nature and potential importance of each variable rather than using a single imputation strategy for the entire dataset.

For numerical variables, median imputation will be used because several financial variables are highly skewed and the median is less sensitive to extreme values than the mean.

For important variables with substantial missingness, missing-value indicators will also be created. This allows the predictive models to determine whether the absence of information itself contains useful information about default risk.

Categorical missing values will be represented as an explicit "Missing" category where appropriate, rather than automatically assuming that the most common category applies. """)

In [ ]:
# Create missingness indicators for important variables

missing_indicator_features = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "YEARS_EMPLOYED",
    "OCCUPATION_TYPE"
]

for feature in missing_indicator_features:
    model_data[f"{feature}_MISSING"] = model_data[feature].isna().astype(int)

print("Missingness indicators created:")
print([f"{feature}_MISSING" for feature in missing_indicator_features])

In [ ]:
# Check missingness indicators

missing_indicators = [
    f"{feature}_MISSING"
    for feature in missing_indicator_features
]

model_data[missing_indicators].sum().sort_values(ascending=False)

In [ ]:
print("""## 4. Train–Validation Split

The dataset is divided into training and validation subsets before imputation and model preparation. A stratified split is used because the target variable is highly imbalanced, with default cases representing approximately 8% of observations.

Stratification ensures that both subsets maintain a similar proportion of default and non-default cases.

The validation dataset will remain untouched during the data preparation process. Imputation parameters will be estimated using the training data only and subsequently applied to the validation data. This prevents information from the validation set from leaking into the training process. """)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = model_data.drop(columns=["TARGET", "SK_ID_CURR"])
y = model_data["TARGET"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training observations:", X_train.shape[0])
print("Validation observations:", X_valid.shape[0])

In [ ]:
print("Overall default rate:")
print(y.mean())

print("\nTraining default rate:")
print(y_train.mean())

print("\nValidation default rate:")
print(y_valid.mean())

In [ ]:
print("""## 5. Missing Value Imputation

Missing values are imputed after the train–validation split to prevent data leakage.

For numerical variables, the median of each variable is calculated using the training data only. These training medians are then used to impute both the training and validation datasets.

For categorical variables, missing observations are replaced with an explicit "Missing" category. This preserves the possibility that missingness itself contains information relevant to credit risk. """)

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

In [ ]:
# Calculate medians using TRAINING DATA ONLY
train_medians = X_train[numeric_features].median()

# Apply the training medians to both datasets
X_train[numeric_features] = X_train[numeric_features].fillna(train_medians)
X_valid[numeric_features] = X_valid[numeric_features].fillna(train_medians)

print("Remaining numerical missing values in training:")
print(X_train[numeric_features].isnull().sum().sum())

print("\nRemaining numerical missing values in validation:")
print(X_valid[numeric_features].isnull().sum().sum())

In [ ]:
# Replace missing categorical values with an explicit category

X_train[categorical_features] = X_train[categorical_features].fillna("Missing")
X_valid[categorical_features] = X_valid[categorical_features].fillna("Missing")

print("Remaining categorical missing values in training:")
print(X_train[categorical_features].isnull().sum().sum())

print("\nRemaining categorical missing values in validation:")
print(X_valid[categorical_features].isnull().sum().sum())

In [ ]:
print("Training set missing values:", X_train.isnull().sum().sum())
print("Validation set missing values:", X_valid.isnull().sum().sum())

print("\nTraining shape:", X_train.shape)
print("Validation shape:", X_valid.shape)

print("\nTraining default rate:", y_train.mean())
print("Validation default rate:", y_valid.mean())

In [ ]:
## Export Prepared Modelling Data
# Save the prepared training and validation datasets

X_train.to_csv(LOCAL_DATA_DIR / "X_train.csv", index=False)
X_valid.to_csv(LOCAL_DATA_DIR / "X_valid.csv", index=False)

y_train.to_csv(LOCAL_DATA_DIR / "y_train.csv", index=False)
y_valid.to_csv(LOCAL_DATA_DIR / "y_valid.csv", index=False)

print("Prepared modelling datasets saved successfully.")